# 프롬프트 작성 방법 — 출력 형식 지정하기 (JSON 모드 · 형식 지침 · 예시)

수업 필기 `프롬프트_작성_방법` (JSON 형식) · `프롬프트_작성_방법2_01` (출력 형식 지정) 을 실행 가능하도록 정리 · 개선한 노트입니다. (강의 코드 `프롬프트_작성_방법2_강의본` 과 대조)

- 📖 교재 : `AI Agent` 교재10 **p.12~14** (프롬프트 엔지니어링 — 중요성 · 효과적인 프롬프트 작성 방법 · 실습)
- 🔗 앞 노트 : [`openai_api_basics.ipynb`](./openai_api_basics.ipynb) (API 기초 · 토큰) · 다음 노트 : [`review_sentiment_analysis.ipynb`](./review_sentiment_analysis.ipynb)
- 📝 원본에서 고친 곳은 그 자리에 **📝 고친 부분** 으로 적고 맨 끝 표에 모았습니다. 그림은 [`images/`](./images/) 폴더

> **출력을 읽는 법** — `# 📼` 셀은 **수업 시간에 실제로 실행한 결과**(필기 원본의 출력)를 옮긴 것, `# ⏳` 셀은 API 키가 필요해 **출력이 비어 있는** 셀(직접 실행), 표시가 없는 셀은 이 노트에서 실제로 실행한 결과입니다.

## 0. 준비

`pip install openai python-dotenv pandas` · 노트북 폴더(또는 상위 폴더)에 `.env` (`OPENAI_API_KEY="sk-…"`) — 자세한 내용은 [`openai_api_basics.ipynb`](./openai_api_basics.ipynb) 0절

In [1]:
# 필요한 함수 / 클래스 임포트
import os, re, json
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

# .env → API 키 (키 값은 출력하지 않는다)
print('.env 읽음 :', load_dotenv(find_dotenv(usecwd=True)), '| 키 있음 :', bool(os.getenv('OPENAI_API_KEY')))

# OpenAI 서버와의 연결 통로 생성
client = OpenAI()

.env 읽음 : True | 키 있음 : True


## 1. 효과적인 프롬프트 작성 방법 (교재 p.12~13)

**프롬프트** = 모델에게 주는 명령 · 질문. 같은 모델이라도 프롬프트에 따라 결과가 달라진다.

| 원칙 (교재 p.13) | 뜻 | 오늘 실습에서 쓴 곳 |
|---|---|---|
| 구체적인 지시 | 무엇을 · 몇 개 · 어떤 조건으로 | "핵심 메시지 **3가지**", "3개월 · 월 200개" |
| 역할 부여 | 시스템 프롬프트로 전문가 역할 | "친환경 · 웰빙 분야 디지털 마케팅 전문가" |
| 맥락 제시 | 배경 정보 | `<핵심 목표>` · `<제품 정보>` |
| 예시 활용 | 원하는 결과의 모양을 보여 준다 | `<예시>` · JSON 키 이름 |
| 반복 및 개선 | 결과를 보고 프롬프트를 고친다 | 3절 — 결과가 형식을 지켰는지 확인 → 프롬프트 수정 |

## 2. 실행 함수

필기는 노트북마다 시스템 프롬프트를 함수 안에 적어 두었다. 시스템 프롬프트와 JSON 모드를 **인자로** 받게 해서 함수 하나로 쓴다.

In [2]:
def get_completion(prompt, system='너는 사용자의 질문에 정확하게 답하는 도우미야.', model='gpt-5-nano', json_mode=False):
    """Chat Completions API 를 한 번 호출하고 답 글자만 돌려준다.

    prompt    : 사용자 프롬프트
    system    : 시스템 프롬프트(역할)
    json_mode : True 면 모델이 반드시 JSON 문자열로 답한다
    """
    options = {}
    if json_mode:
        options['response_format'] = {'type': 'json_object'}    # JSON 모드
    response = client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system', 'content': system},   # 시스템 프롬프트 : 역할 · 규칙
            {'role': 'user', 'content': prompt},     # 사용자 프롬프트 : 요청
        ],
        **options,
    )
    return response.choices[0].message.content

## 3. 방법 1 — JSON 형식으로 받기 (필기 `프롬프트_작성_방법`)

결과를 **프로그램이 이어서 쓰려면** 글이 아니라 정해진 모양(JSON)이어야 한다.

- `response_format={'type': 'json_object'}` (**JSON 모드**) : 답이 반드시 올바른 JSON 문자열
- 조건 : 프롬프트 어딘가에 **"JSON"** 이라는 말이 있어야 한다(없으면 요청이 거절된다)
- JSON 모드가 보장하는 것은 **문법**뿐이다. 어떤 키를 쓸지는 프롬프트에 적어야 한다

In [3]:
# 📼 아래 출력 = 수업 시간에 실행한 결과 (필기 원본에서 옮김)
# 사용자 프롬프트 : JSON 형식으로 응답 생성 (필기 원본 그대로)
books_prompt = """
3개의 가상의 책 제목, 저자, 장르를 생성해 줘.
아래 키를 사용하여 json 형식으로 제공해야되
book_id,title,author,genre
"""
output = get_completion(books_prompt, system='너는 사용자의 답변에 말해주는 도우미야.', json_mode=True)
print(f'LLM 응답:{output}')

LLM 응답:{
  "books": [
    { "book_id": "B001", "title": "달빛 아래의 서약", "author": "이지은", "genre": "Fantasy" },
    { "book_id": "B002", "title": "비밀의 도서관", "author": "박수진", "genre": "Mystery" },
    { "book_id": "B003", "title": "시간을 걷는 로봇", "author": "김도훈", "genre": "Science Fiction" }
  ]
}


In [4]:
# 받은 것은 '문자열' 이다 → json.loads 로 dict 로 바꿔야 값을 꺼낼 수 있다
print(type(output).__name__, len(output), '자')

data = json.loads(output)                 # JSON 문자열 → dict
print(type(data).__name__, '| 최상위 키 :', list(data.keys()))

books = pd.DataFrame(data['books'])       # 책 목록 → 표
books

str 283 자
dict | 최상위 키 : ['books']


,book_id,title,author,genre
0,B001,달빛 아래의 서약,이지은,Fantasy
1,B002,비밀의 도서관,박수진,Mystery
2,B003,시간을 걷는 로봇,김도훈,Science Fiction


- 문자열 → `json.loads` → `dict` → `DataFrame`. 여기서부터는 평소의 파이썬 데이터다
- 프롬프트에 없는 것은 **모델이 정했다** : 최상위 키 `books`, `book_id` 의 모양(`B001`), 장르는 **영어**(`Fantasy`)
  - JSON 모드의 답은 `{ }` 객체 하나여야 해서 목록을 감쌀 키가 필요했고, 이름을 안 정해 줬으니 모델이 `books` 로 지었다. 다시 실행하면 `items` · `data` 로 바뀔 수 있다 → `data['books']` 가 `KeyError`
- 그래서 **키 이름 · 값의 형식 · 언어**까지 적는다

> **📝 고친 부분**
> - 최상위 키와 값의 형식이 프롬프트에 없음 → 아래처럼 예시 JSON 으로 고정
> - 시스템 프롬프트 `너는 사용자의 답변에 말해주는 도우미야.` → '사용자의 **질문에 답하는**' (말이 뒤바뀜)
> - `제공해야되` → 제공해야 돼 · 제목 `chat comletion` → completion

In [5]:
# 고친 프롬프트 — 예시 JSON 은 json.dumps 로 만든다 (큰따옴표 보장)
books_example = {'books': [{'book_id': 1, 'title': '책 제목', 'author': '저자 이름', 'genre': '장르'}]}

books_prompt_v2 = f"""
가상의 책 3권의 제목, 저자, 장르를 만들어 줘.

출력 형식:
- 아래 <예시> 와 똑같은 키를 가진 JSON 으로만 답한다.
- 최상위 키는 "books" 하나이고, 값은 책 3권의 목록이다.
- book_id 는 1부터 시작하는 정수, title · author · genre 는 한국어 문자열이다.

<예시>
{json.dumps(books_example, ensure_ascii=False, indent=2)}
"""
print(books_prompt_v2)


가상의 책 3권의 제목, 저자, 장르를 만들어 줘.

출력 형식:
- 아래 <예시> 와 똑같은 키를 가진 JSON 으로만 답한다.
- 최상위 키는 "books" 하나이고, 값은 책 3권의 목록이다.
- book_id 는 1부터 시작하는 정수, title · author · genre 는 한국어 문자열이다.

<예시>
{
  "books": [
    {
      "book_id": 1,
      "title": "책 제목",
      "author": "저자 이름",
      "genre": "장르"
    }
  ]
}



In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)
output_v2 = get_completion(books_prompt_v2, json_mode=True)
data_v2 = json.loads(output_v2)
pd.DataFrame(data_v2['books'])       # book_id 가 정수 · 장르가 한국어인지 확인

## 4. 방법 2 — 형식 지침과 예시로 글의 모양 정하기 (필기 `프롬프트_작성_방법2`)

보고서처럼 **사람이 읽을 글**은 JSON 대신 '어떤 제목 · 어떤 목록으로 쓸지' 를 지침과 예시로 정한다. 생분해성 요가 매트의 마케팅 전략을 요청하는 프롬프트의 구조 :

| 부분 | 내용 | 원칙 |
|---|---|---|
| 시스템 프롬프트 | 친환경 · 웰빙 분야 디지털 마케팅 전문가 | 역할 부여 |
| `<핵심 목표>` · `<제품 정보>` | 3개월 · 월 200개 · 예산 월 300만 원 · 타겟 고객 | 맥락 제시 (`< >` 구분자로 지시와 자료를 나눔) |
| 포함할 항목 1 · 2 | 핵심 메시지 3가지 · 단계별 방향 · 타겟 고객 분석 | 구체적인 지시 |
| ```` ```형식 지침``` ```` | 리스트(-) 또는 문단 · **서문 / 맺음말 없이 요청된 내용만** | 출력 형식 |
| `<예시>` | `1. 전략 개요 …` / `2. 타겟 고객 …` 의 뼈대 | 예시 활용 |

In [6]:
# 사용자 프롬프트 (강의 코드 그대로 — 아래 수업 실행 결과와 같은 프롬프트)
user_prompt = """
< >로 구분된 핵심 목표와 제품 정보를 바탕으로 '생분해성 요가 매트'의 성공적인 시장 안착을 위한 디지털 마케팅 전략을 수립하여 제시하십시오.

<핵심 목표>
3개월 내 월 판매량 200개 달성 및 친환경 요가 매트 시장 내 브랜드 인지도 확립

<제품 정보>
1.제품: 생분해성 요가 매트
2.핵심 가치: 친환경, 지속 가능성, 뛰어난 그립감과 편안함, 유해 물질 무첨가
3.타겟 고객: 환경 보호에 관심이 많고 요가를 즐겨 하는 20대 후반 ~ 40대 중반 여성
4.타겟 고객의 특징: 온라인 정보 탐색에 능숙하며, 가치 소비를 지향하고, 건강과 웰빙에 투자하는 경향이 있음.
5.월별 예산: 300만원 (총 3개월간 900만원)
6.기간: 3개월
7.주요 목표: 3개월 내 월 판매량 200개 달성 및 친환경 요가 매트 시장 내 브랜드 인지도 확립

제시하는 전략은 다음 항목들을 반드시 포함해야 합니다. 각 항목은 매우 구체적이고 즉시 실행 가능한 내용으로 구성되어야
하며, 단순한 나열을 넘어 각 전략의 핵심 의도와 기대 효과를 명확히 설명해 주세요.

1.  전략 개요 및 핵심 메시지:
    * 제품의 친환경적 가치와 타겟 고객의 니즈를 연결하는 핵심 메시지 3가지를 명확히 제시하십시오.
    * 3개월 전략의 전반적인 방향성을 단계별로 요약하십시오 (예: 인지도 확산 → 구매 유도 → 재구매 및 충성도 강화).

2.  타겟 고객 상세 분석:
    * 타겟 고객의 특성, 온라인 활동 패턴, 구매 동기, Pain Points 등을 심층적으로 분석하여 제시하십시오.

출력 형식:
다음 ```형식 지침```과 <예시>를 철저히 준수하여 응답을 작성하십시오.

```형식 지침```
각 섹션은 <예시>와 같이 가독성 높은 리스트(-) 또는 문단으로 구성해 주세요.
불필요한 서문이나 맺음말 없이 요청된 내용만을 작성하십시오.

<예 시>
1. 전략 개요 및 핵심 메시지
- [핵심 메시지 1]
- [핵심 메시지 2]
- 전반적인 전략 방향 요약: [내용]
2. 타겟 고객 상세 분석
- [특성 1]
- [특성 2]
- [온라인 활동 패턴]
- [Pain Points]
- [제품을 통한 해결책]
"""
print(len(user_prompt), '자')

1075 자


In [7]:
# 📼 아래 출력 = 수업 시간에 실행한 결과 (필기 원본에서 옮김)
# 수업 때의 시스템 프롬프트 그대로 ("월빙" 은 "웰빙" 의 오타 — 4-1 뒤에서 고침)
system_marketing = '너는 "친환경/지속가능성"과 "월빙" 분야를 전문으로 하는 최고 수준의 디지털 마케팅 전문가야.'

output = get_completion(user_prompt, system=system_marketing)
print(f"LLM 출력: \n{output}")

LLM 출력: 
1. 전략 개요 및 핵심 메시지
- [핵심 메시지 1] 환경을 생각하는 당신의 요가, 지구에 작은 변화로 큰 여정을 시작하세요.
  - 의도: 친환경 가치와 요가 몰입감을 연결하여 환경 의식이 높은 소비자의 공감과 브랜드 신뢰를 확보
  - 기대 효과: 브랜드 신뢰도 증가, 친환경 가치에 대한 인지도 상승, 관심도(CTR/조회수) 향상

- [핵심 메시지 2] 무해 물질 0%로 안전하게, 뛰어난 그립과 편안함으로 자세에 집중하세요.
  - 의도: 제품의 안전성·성능 차별화 강조
  - 기대 효과: 안전성에 민감한 고객의 구매 의사 결정 가속, 클릭률 및 제품 비교 관심 증가

- [핵심 메시지 3] 당신의 가치 소비를 반영하는 프리미엄 친환경 매트
  - 의도: 프리미엄 포지셔닝으로 가치 소비 성향의 타깃에 어필
  - 기대 효과: 가격대가 높은 친환경 카테고리에서의 전환율 상승, 브랜드 충성도 형성

- 전반적인 전략 방향 요약: [3개월은 인지도 확산 → 구매 유도 → 재구매 및 충성도 강화의 단계적 흐름으로 운영합니다.]
  - 1단계(1–4주): 인지도 확산 및 신뢰 구축
    - 실행: 메타/구글 네이티브 광고로 친환경 가치 스토리와 제품 특성 홍보, 마이크로 인플루언서 및 UGC 활용, SEO 중심의 EDUCATION 콘텐츠(생분해성 원리, 재료 비교, 안전성) 생산
  - 2단계(5–9주): 구매 유도 및 트래픽 전환
    - 실행: 리타겟팅 및 프로모션 코드·번들 패키지 제안, 비교 콘텐츠(타사 일반 매트와의 차별점) 제공, 리뷰/비교 영상 확산
  - 3단계(10–12주): 재구매 및 충성도 강화
    - 실행: 이메일 마케팅과 회원 프로그램 도입, 재활용/리사이클링 인센티브, 구매 후 피드백 및 UGC 참여 유도
  - 예산 배분(총 9,000,000원): 광고 4,000,000원, 인플루언서/UGC 2,000,000원, 콘텐츠 제작 및 SEO 1,000,000원, 랜딩/CRM 1,000,000원, 예비 및 테스트 1,0

### 4-1. 형식을 지켰는지 확인 — 눈으로 말고 코드로

지침은 **"불필요한 서문이나 맺음말 없이 요청된 내용만"** 이고 예시의 섹션은 `1.` · `2.` 두 개였다. 들여쓰기 없이 시작하고 `-` 로 시작하지 않는 줄이 **섹션 제목**이므로 그것을 세어 본다.

In [8]:
lines = output.splitlines()

# 섹션 제목 = 내용이 있고, 공백이나 '-' 로 시작하지 않는 줄
heads = [(i, line) for i, line in enumerate(lines) if line and not line.startswith((' ', '-'))]

rows = []
for k, (i, title) in enumerate(heads):
    end = heads[k + 1][0] if k + 1 < len(heads) else len(lines)        # 다음 제목 전까지가 이 섹션
    body = '\n'.join(lines[i:end])
    requested = title.startswith(('1.', '2.'))                          # 프롬프트가 요청한 섹션인가
    rows.append({'섹션 제목': title[:28], '글자 수': len(body), '요청한 섹션': 'O' if requested else 'X'})

check = pd.DataFrame(rows)
total = check['글자 수'].sum()
extra = check.loc[check['요청한 섹션'] == 'X', '글자 수'].sum()
print(f'요청하지 않은 부분 : {extra}자 / {total}자 = {extra / total:.0%}')
check

요청하지 않은 부분 : 803자 / 3105자 = 26%


,섹션 제목,글자 수,요청한 섹션
0,1. 전략 개요 및 핵심 메시지,1075,O
1,2. 타겟 고객 상세 분석,1227,O
2,추가 실행 포인트,632,X
3,참고: 본 전략은 3개월이라는 짧은 기간 동안 월,171,X


In [9]:
# 그 밖의 확인
msgs = [l for l in lines if l.startswith('- [핵심 메시지')]
print('핵심 메시지 개수            :', len(msgs), '(지시 3가지 · 예시에는 2개만 있었음)')
print("예시의 대괄호를 그대로 쓴 줄 :", sum(l.startswith('- [') for l in lines), '개  예)', msgs[0][:22], '…')
print('맺음말로 끝났는가           :', lines[-1][:16], '…')

핵심 메시지 개수            : 3 (지시 3가지 · 예시에는 2개만 있었음)
예시의 대괄호를 그대로 쓴 줄 : 8 개  예) - [핵심 메시지 1] 환경을 생각하는  …
맺음말로 끝났는가           : 참고: 본 전략은 3개월이라는 …


- `1.` · `2.` 두 섹션의 뼈대는 예시대로 나왔고, 핵심 메시지는 예시(2개)가 아니라 **지시(3가지)** 를 따랐다 → 예시와 지시가 다르면 지시가 이긴 경우
- 그런데 요청하지 않은 **`추가 실행 포인트`** 와 **`참고:` 맺음말**이 붙었다 — 전체의 약 4분의 1. "맺음말 없이 요청된 내용만" 을 어겼다
- 예시의 `[핵심 메시지 1]` 은 '여기에 내용을 넣어라' 는 자리 표시였는데, 모델은 **대괄호까지 그대로** 제목처럼 썼다

![형식 준수 확인 애니메이션](./images/anim_05_format_check.gif)

원인으로 볼 만한 프롬프트의 틈 :

| 틈 | 왜 문제인가 |
|---|---|
| "매우 구체적이고 **즉시 실행 가능한** 내용 · **의도와 기대 효과**를 설명" ↔ "요청된 내용만" | 앞의 지시가 '더 써라' 로 읽힌다. 모델은 실행 포인트를 덧붙여 구체성을 채웠다 |
| 지침은 `<예시>` 를 따르라는데 실제 태그는 `<예 시>` (띄어쓰기) · 닫는 태그 없음 | 어디까지가 예시인지 흐리다 |
| `[핵심 메시지 1]` 이 자리 표시라는 설명이 없음 | 글자 그대로 따라 씀 |
| 금지가 "불필요한" 이라는 **판단이 필요한 말** | 모델은 추가 내용을 '필요하다' 고 판단할 수 있다 |
| `<핵심 목표>` 와 제품 정보 7번이 같은 문장 | 토큰만 더 든다 |

> **📝 고친 부분** — 아래 `user_prompt_v2` : 닫는 태그 · 자리 표시 설명 · "섹션은 정확히 2개, 그 밖의 제목 · 요약 · 참고 문장 금지" 처럼 **셀 수 있는 규칙**으로 바꿈. 시스템 프롬프트 `월빙` → `웰빙`.

In [10]:
system_marketing_v2 = '너는 "친환경/지속가능성"과 "웰빙" 분야를 전문으로 하는 최고 수준의 디지털 마케팅 전문가야.'

user_prompt_v2 = """
<핵심 목표> 와 <제품 정보> 를 바탕으로 '생분해성 요가 매트' 의 시장 안착을 위한 디지털 마케팅 전략을 작성하십시오.

<핵심 목표>
3개월 내 월 판매량 200개 달성 및 친환경 요가 매트 시장 내 브랜드 인지도 확립
</핵심 목표>

<제품 정보>
1. 제품: 생분해성 요가 매트
2. 핵심 가치: 친환경, 지속 가능성, 뛰어난 그립감과 편안함, 유해 물질 무첨가
3. 타겟 고객: 환경 보호에 관심이 많고 요가를 즐겨 하는 20대 후반 ~ 40대 중반 여성
4. 타겟 고객의 특징: 온라인 정보 탐색에 능숙하며, 가치 소비를 지향하고, 건강과 웰빙에 투자하는 경향이 있음
5. 예산: 월 300만원 (3개월 총 900만원)
</제품 정보>

<작성 항목>
1. 전략 개요 및 핵심 메시지
   - 제품의 친환경 가치와 타겟 고객의 니즈를 연결하는 핵심 메시지 3가지 (메시지마다 의도와 기대 효과를 한 줄씩)
   - 3개월 전략 방향을 단계별로 요약 (예: 인지도 확산 → 구매 유도 → 재구매 및 충성도 강화)
2. 타겟 고객 상세 분석
   - 특성, 온라인 활동 패턴, 구매 동기, Pain Points, 제품을 통한 해결책
</작성 항목>

<형식 지침>
- 섹션은 <예시> 의 1번, 2번 정확히 2개만 쓴다. 그 밖의 제목, 추가 제안, 요약, 참고 문장, 서문, 맺음말은 쓰지 않는다.
- 각 섹션은 리스트(-)로 쓰고, 하위 항목은 두 칸 들여 쓴다.
- <예시> 의 대괄호 [ ] 는 내용을 채울 자리 표시다. 출력에는 대괄호를 쓰지 않는다.
- 마지막 줄은 2번 섹션의 마지막 항목이어야 한다.
</형식 지침>

<예시>
1. 전략 개요 및 핵심 메시지
- 핵심 메시지 1: [메시지]
  - 의도: [내용]
  - 기대 효과: [내용]
- 전략 방향 요약: [내용]
2. 타겟 고객 상세 분석
- 특성: [내용]
- 온라인 활동 패턴: [내용]
- 구매 동기: [내용]
- Pain Points: [내용]
- 제품을 통한 해결책: [내용]
</예시>
"""
print(f'원본 {len(user_prompt)}자 → 고친 프롬프트 {len(user_prompt_v2)}자')

원본 1075자 → 고친 프롬프트 1000자


In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)
# 고친 프롬프트로 다시 실행 → 4-1 의 확인 코드를 그대로 돌려 '요청하지 않은 부분' 이 0% 가 되는지 본다
output_v2 = get_completion(user_prompt_v2, system=system_marketing_v2)
print(output_v2)

lines_v2 = output_v2.splitlines()
heads_v2 = [l for l in lines_v2 if l and not l.startswith((' ', '-'))]
print('\n섹션 제목 :', heads_v2)
print('대괄호 줄 :', sum('[' in l for l in lines_v2))

같은 프롬프트라도 실행할 때마다 답이 조금씩 다르다. 한 번 지켰다고 끝이 아니라 **여러 번 돌려 확인 코드로 세는 것**이 '반복 및 개선'(교재 p.13)이다.

## 5. 내 필기 `프롬프트_작성_방법2_01` 이 실행되지 않은 이유

필기 노트북은 마지막 셀이 실행되지 않은 채였다. 원본 코드를 그대로 옮겨 어디서 막히는지 차례로 본다.

In [11]:
# 필기 원본 그대로의 함수
def get_completion_note(prompt, model='get-5-nano'):          # ① 모델 이름 오타 : get-5-nano
    response = client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system', 'content': '너는 "친환경/지속가능성"과 "웰빙" 분야를 전문으로 하는 최고 수준의 디지털 마케팅 전문이다.'},
            {'role': 'user', 'content': promt},                # ② 변수 이름 오타 : promt
        ],
    )
    return response.choices[0].message.content

# 호출 1 — 필기 그대로 : 인자 이름도 promt
try:
    output_note = get_completion_note(promt=user_prompt)
except TypeError as e:
    print('TypeError :', e)

# 호출 2 — 인자 이름만 고쳐도 함수 안의 promt 에서 막힌다
try:
    output_note = get_completion_note(prompt=user_prompt)
except NameError as e:
    print('NameError :', e)

# 출력문 — f-string 인데 중괄호가 아니라 소괄호
output_note = '(모델의 답)'
print(f"LLM 출력:(output_note)")
print(f"LLM 출력:\n{output_note}")

TypeError : get_completion_note() got an unexpected keyword argument 'promt'. Did you mean 'prompt'?
NameError : name 'promt' is not defined
LLM 출력:(output_note)
LLM 출력:
(모델의 답)


오류는 **한 번에 하나씩** 보인다. 하나를 고치면 다음 것이 나타난다.

| 순서 | 원본 | 결과 | 고침 |
|---|---|---|---|
| 1 | `get_completion(promt=user_prompt)` | `TypeError` — 함수에 `promt` 라는 인자가 없다 | `prompt=` |
| 2 | 함수 안 `'content': promt` | `NameError` — 정의되지 않은 이름 | `prompt` |
| 3 | `model='get-5-nano'` | 1 · 2 를 고치면 요청이 서버까지 가고, **없는 모델**이라 서버가 404(`NotFoundError`)로 거절 (※ 이 줄만 서버 응답이라 여기서는 재현하지 못함) | `gpt-5-nano` |
| 4 | `print(f"LLM 출력:(output)")` | 오류 없이 `(output)` 글자가 그대로 찍힌다 | `{output}` |

1 · 2 는 파이썬이 요청을 **보내기 전에** 잡아 주고(요금 없음), 3 은 서버가 잡아 준다. 4 는 아무도 잡아 주지 않는다 — 출력을 직접 봐야 한다.

## 📝 고친 부분 정리

| 위치 | 필기 원본 | 고친 내용 |
|---|---|---|
| 방법2 함수 | `model='get-5-nano'` | `gpt-5-nano` |
| 방법2 함수 | `'content': promt` · `get_completion(promt=…)` | `prompt` |
| 방법2 출력 | `print(f"LLM 출력:(output)")` | `{output}` |
| 방법2 시스템 프롬프트 | `디지털 마케팅 전문이다` · (강의 코드) `월빙` | 전문가야 · 웰빙 |
| 방법2 프롬프트 | `<예시>` 를 따르라면서 태그는 `<예 시>` · 닫는 태그 없음 · 목표 문장 중복 | 여는 / 닫는 태그 통일 · 중복 삭제 |
| 방법2 프롬프트 | "불필요한 서문이나 맺음말 없이" → 실제 결과에 `추가 실행 포인트` · `참고:` 가 붙음 | "섹션은 정확히 2개 · 마지막 줄은 2번 섹션의 항목" 처럼 셀 수 있는 규칙 + 확인 코드 |
| 방법2 프롬프트 | 예시의 `[핵심 메시지 1]` 이 그대로 출력됨 | 대괄호는 자리 표시라고 명시 |
| 방법1 프롬프트 | 최상위 키 · 값 형식 · 언어 미지정 (`books` · `B001` · `Fantasy` 는 모델이 정함) | 예시 JSON(`json.dumps`)으로 고정 |
| 방법1 시스템 프롬프트 | `사용자의 답변에 말해주는 도우미` | 사용자의 질문에 답하는 도우미 |
| 방법1 결과 | 문자열을 `print` 만 함 | `json.loads` → `DataFrame` 까지 |
| 함수 | 노트북마다 시스템 프롬프트를 함수 안에 고정 | `system` · `json_mode` 인자 |
| 환경 설정 | `r"C:\Users\Jinho\work\.env"` 절대 경로 | `find_dotenv()` |
| 오타 | `comletion` · `제공해야되` · `#모델 생성`(실제로는 클라이언트 생성) | completion · 제공해야 돼 · 연결 통로(클라이언트) 생성 |